In [25]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split


from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, Dropout, GRU , Bidirectional
from tensorflow.keras.callbacks import EarlyStopping

import warnings
warnings.filterwarnings("ignore")

In [26]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

train_df.shape

(104632, 4)

In [27]:
test_df.shape

(17373, 4)

In [28]:
train_df.columns

Index(['q_id', 'eng', 'class', 'chapter'], dtype='object')

In [29]:
test_df.columns

Index(['q_id', 'eng', 'class', 'chapter'], dtype='object')

In [30]:
df = pd.concat([train_df,test_df],ignore_index=True)
df.drop("q_id",axis=1,inplace=True)
df.head()

,eng,class,chapter
0,Photodiode is a device\nA. Which is always ope...,12,"SEMICONDUCTOR ELECTRONICS: MATERIALS, DEVICES ..."
1,When water is heated from \( 0^{\circ} \mathrm...,11,THERMAL PROPERTIES OF MATTER
2,Potentiometer measures the potential differenc...,12,CURRENT ELECTRICITY
3,"In an isosceles trapezium, the length of one o...",12,APPLICATION OF INTEGRALS
4,The normal to the curve \( x^{2}=4 y \)\npassi...,12,APPLICATION OF DERIVATIVES


In [31]:
df["class"].unique()

array([12, 11, 10,  9,  7,  8,  6])

In [32]:
# def labalencoder(class_df):
#     if(6 == class_df):
#         return 1
#     elif(7 == class_df):
#         return 2
#     elif(8 == class_df):
#         return 3
#     elif(9 == class_df):
#         return 4
#     elif(10 == class_df):
#         return 5
#     elif(11 == class_df):
#         return 6
#     else:
#         return 7
    
# df["class"] = df["class"].apply(labalencoder)

# df.head()

In [33]:
df["chapter"].unique()

array(['SEMICONDUCTOR ELECTRONICS: MATERIALS, DEVICES AND SIMPLE CIRCUITS',
       'THERMAL PROPERTIES OF MATTER', 'CURRENT ELECTRICITY',
       'APPLICATION OF INTEGRALS', 'APPLICATION OF DERIVATIVES',
       'CONTINUITY AND DIFFERENTIABILITY', 'ELECTROMAGNETIC INDUCTION',
       'STRAIGHT LINES', 'STATISTICS',
       'PAIR OF LINEAR EQUATIONS IN TWO VARIABLES',
       'ANATOMY OF FLOWERING PLANTS',
       'GENERAL PRINCIPLES AND PROCESSES OF ISOLATION OF ELEMENTS',
       'THE p-BLOCK ELEMENTS', 'UNITS AND MEASUREMENT', 'NUMBER SYSTEMS',
       'LIFE PROCESSES', 'REDOX REACTIONS', 'ELECTRIC CHARGES AND FIELDS',
       'STRUCTURE OF ATOM', 'COORDINATION COMPOUNDS',
       'MOVING CHARGES AND MAGNETISM', 'ATOMS', 'BIOMOLECULES',
       'WORK, ENERGY AND POWER', 'NUCLEI', 'LIMITS AND DERIVATIVES',
       'CLASSIFICATION OF ELEMENTS AND PERIODICITY IN PROPERTIES',
       'CHEMICAL KINETICS', 'ENVIRONMENTAL CHEMISTRY', 'VECTOR ALGEBRA',
       'WINDS, STORMS AND CYCLONES', 'THE d-AND f-BL

In [34]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 122005 entries, 0 to 122004
Data columns (total 3 columns):
 #   Column   Non-Null Count   Dtype 
---  ------   --------------   ----- 
 0   eng      122005 non-null  object
 1   class    122005 non-null  int64 
 2   chapter  122005 non-null  object
dtypes: int64(1), object(2)
memory usage: 2.8+ MB


In [35]:
df.describe()

,class
count,122005.000000
mean,11.091652
std,1.184980
min,6.000000
25%,11.000000
50%,11.000000
75%,12.000000
max,12.000000


In [36]:
df.duplicated().sum()

np.int64(590)

In [37]:
df.drop_duplicates(inplace=True)
df.duplicated().sum()

np.int64(0)

In [38]:
df.isnull().sum()

eng        0
class      0
chapter    0
dtype: int64

In [39]:
def tolower(txt):
    return txt.lower()

def remove_punctuation(txt):
    import string
    return txt.translate(str.maketrans("", "", string.punctuation))

def remove_emojis(txt):
    new_txt = ""
    for i in txt:
        if i.isascii():
            new_txt+=i
    return new_txt

In [40]:
df["eng"] = df["eng"].apply(tolower)
df["eng"] = df["eng"].apply(remove_punctuation)
df["eng"] = df["eng"].apply(remove_emojis)
df["chapter"] = df["chapter"].apply(tolower)
df["chapter"] = df["chapter"].apply(remove_punctuation)
df["chapter"] = df["chapter"].apply(remove_emojis)

df.head()

,eng,class,chapter
0,photodiode is a device\na which is always oper...,12,semiconductor electronics materials devices an...
1,when water is heated from 0circ mathrmc its\...,11,thermal properties of matter
2,potentiometer measures the potential differenc...,12,current electricity
3,in an isosceles trapezium the length of one of...,12,application of integrals
4,the normal to the curve x24 y \npassing throu...,12,application of derivatives


In [41]:
df["eng"] = df["eng"] + "" + df["chapter"]
df.drop("chapter",axis=1,inplace=True)
df.head()

,eng,class
0,photodiode is a device\na which is always oper...,12
1,when water is heated from 0circ mathrmc its\...,11
2,potentiometer measures the potential differenc...,12
3,in an isosceles trapezium the length of one of...,12
4,the normal to the curve x24 y \npassing throu...,12


In [42]:
df["eng"][2]

'potentiometer measures the potential difference more accurately than a voltmeter because\na it has a wire of high resistance\nb it has a wire of low resistance\nc it does not draw current from external circuit\nd it draws a heavy current from external circuitcurrent electricity'

In [43]:
X = df.drop("class",axis=1)
y = df[["class"]]

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [44]:
Max_len = 100
Max_words = 10000
Embedding_dim = 100

In [45]:
tokenizer = Tokenizer(num_words=5000, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train["eng"])
train_seq = tokenizer.texts_to_sequences(X_train["eng"])
train_X_pad = pad_sequences(train_seq, maxlen=Max_len, padding="post", truncating="post")
test_seq = tokenizer.texts_to_sequences(X_test["eng"])
test_X_pad = pad_sequences(test_seq, maxlen=Max_len, padding="post",truncating="post")

In [46]:
len(df["class"].unique())

7

In [47]:
model = Sequential([
    Embedding(input_dim=Max_words, output_dim=Embedding_dim, input_length=Max_len),
    Bidirectional(GRU(128,return_sequences=True)),
    Dropout(0.3),
    Bidirectional(GRU(64,return_sequences=False)),
    Dropout(0.2),
    Dense(32, activation="relu"),
    Dense(16, activation="relu"),
    Dense(1),
])
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [48]:
model.compile(loss="mse", optimizer="adam", metrics=["r2_score"])
model.EarlyStopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
his = model.fit(train_X_pad, y_train, validation_data=(test_X_pad, y_test), epochs=2, batch_size=128)
model.summary()

Epoch 1/2
759/759 ━━━━━━━━━━━━━━━━━━━━ 608s 786ms/step - loss: 2.6842 - r2_score: -0.9086 - val_loss: 0.2936 - val_r2_score: 0.7913
Epoch 2/2
759/759 ━━━━━━━━━━━━━━━━━━━━ 513s 676ms/step - loss: 0.4177 - r2_score: 0.7030 - val_loss: 0.2824 - val_r2_score: 0.7993


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 100, 100)       │     1,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, 100, 256)       │       176,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 100, 256)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ (None, 128)            │       123,648 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 32)             │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,914,885 (14.93 MB)

 Trainable params: 1,304,961 (4.98 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2,609,924 (9.96 MB)

In [49]:
model.evaluate(test_X_pad, y_test)

759/759 ━━━━━━━━━━━━━━━━━━━━ 45s 59ms/step - loss: 0.2824 - r2_score: 0.7993


[0.28239697217941284, 0.7992874383926392]

In [50]:
from joblib import dump
model.save("model.h5")
dump(tokenizer,"tokenizer.pkl")
dump(X.columns.tolist(),"columns.pkl")

['columns.pkl']